In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [20]:
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

df['species'] = iris.target

df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

df = df.sample(frac=1).reset_index(drop=True)

df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species,species_name
0,5.8,4.0,1.2,0.2,0,setosa
1,5.1,2.5,3.0,1.1,1,versicolor
2,6.6,3.0,4.4,1.4,1,versicolor
3,5.4,3.9,1.3,0.4,0,setosa
4,7.9,3.8,6.4,2.0,2,virginica
...,...,...,...,...,...,...
145,6.3,2.8,5.1,1.5,2,virginica
146,6.4,3.1,5.5,1.8,2,virginica
147,6.3,2.5,4.9,1.5,1,versicolor
148,6.7,3.1,5.6,2.4,2,virginica


In [21]:
df = df.drop('species_name', axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   species            150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [22]:
data = np.array(df)
m,n = data.shape
data

train_data = data[:100].T
test_data = data[100:150].T

Y_train = train_data[-1].astype(float)
X_train = train_data[1:n].astype(float)

Y_test = test_data[-1].astype(float)
X_test = test_data[1:n].astype(float)

X_train.shape

(4, 100)

In [23]:
def init_params():
    W1 = np.random.rand(10, 4)
    b1 = np.random.rand(10, 1)
    W2 = np.random.rand(10, 10)
    b2 = np.random.rand(10, 1)
    W3 = np.random.rand(3, 10)
    b3 = np.random.rand(3, 1)
    return W1, b1, W2, b2, W3, b3

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A

def forward_prop(W1, b1, W2, b2, W3, b3, X):
    Z1 = np.dot(W1, X) + b1
    A1 = ReLU(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = ReLU(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = softmax(Z3)
    return Z1, A1, Z2, A2, Z3, A3

def one_hot(Y, num_classes=3):
    Y = Y.astype(int)
    one_hot_Y = np.zeros((num_classes, Y.shape[0]), dtype=float)
    one_hot_Y[Y, np.arange(Y.shape[0])] = 1.0
    return one_hot_Y

def ReLU_deriv(Z):
    return (Z > 0).astype(float)


def back_prop(Z1, A1, Z2, A2, Z3, A3, W1, W2, W3, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)

    dZ3 = A3 - one_hot_Y
    dW3 = 1 / m * np.dot(dZ3, A2.T)
    db3 = 1 / m * np.sum(dZ3, axis=1, keepdims=True)

    dZ2 = np.dot(W3.T, dZ3) * ReLU_deriv(Z2)
    dW2 = 1 / m * np.dot(dZ2, A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = np.dot(W2.T, dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * np.dot(dZ1, X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

def update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    W3 = W3 - alpha * dW3
    b3 = b3 - alpha * db3
    return W1, b1, W2, b2, W3, b3

In [24]:
def get_predictions(A3):
    return np.argmax(A3, axis=0)

def get_accuracy(predictions, Y):
    return np.mean(predictions == Y) * 100

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2, W3, b3 = init_params()

    for i in range(iterations):
        np.random.seed(1)
        Z1, A1, Z2, A2, Z3, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X)

        dW1, db1, dW2, db2, dW3, db3 = back_prop(Z1, A1, Z2, A2, Z3, A3, W1, W2, W3, X, Y)

        W1, b1, W2, b2, W3, b3 = update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha)

        if i % 10 == 0:
            predictions = get_predictions(A3)
            acc = get_accuracy(predictions, Y)
            print(f"Epoch: {i} - Accuracy: {acc:.2f}%")

    return W1, b1, W2, b2, W3, b3

In [25]:
np.random.seed(1)
W1,b1,W2,b2,W3,b3 = gradient_descent(X_train,Y_train,0.01,250)

Epoch: 0 - Accuracy: 32.00%
Epoch: 10 - Accuracy: 32.00%
Epoch: 20 - Accuracy: 60.00%
Epoch: 30 - Accuracy: 32.00%
Epoch: 40 - Accuracy: 31.00%
Epoch: 50 - Accuracy: 33.00%
Epoch: 60 - Accuracy: 63.00%
Epoch: 70 - Accuracy: 63.00%
Epoch: 80 - Accuracy: 63.00%
Epoch: 90 - Accuracy: 63.00%
Epoch: 100 - Accuracy: 63.00%
Epoch: 110 - Accuracy: 63.00%
Epoch: 120 - Accuracy: 63.00%
Epoch: 130 - Accuracy: 63.00%
Epoch: 140 - Accuracy: 63.00%
Epoch: 150 - Accuracy: 66.00%
Epoch: 160 - Accuracy: 69.00%
Epoch: 170 - Accuracy: 76.00%
Epoch: 180 - Accuracy: 82.00%
Epoch: 190 - Accuracy: 89.00%
Epoch: 200 - Accuracy: 95.00%
Epoch: 210 - Accuracy: 97.00%
Epoch: 220 - Accuracy: 99.00%
Epoch: 230 - Accuracy: 100.00%
Epoch: 240 - Accuracy: 100.00%


In [26]:
def make_predictions(X, W1, b1, W2, b2, W3, b3):
    np.random.seed(1)
    _,_,_,_,_,A3 = forward_prop(W1,b1,W2,b2,W3,b3,X)
    predictions = get_predictions(A3)
    return predictions

predictions = make_predictions(X_test, W1, b1, W2, b2, W3, b3)
get_accuracy(predictions, Y_test)

np.float64(98.0)

In [27]:
predictions

array([0, 1, 0, 1, 1, 0, 1, 0, 0, 2, 2, 2, 0, 0, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 2, 1, 2, 0, 0, 2, 1, 2,
       1, 2, 2, 1, 2, 0])